# Phase 3A — NumPy/BLAS vs ThreadPoolExecutor (CPU)

Pilot first, then full benchmark. **No timing results exist until you run this notebook.** Same real 20K×384 MiniLM embeddings and top-10 results across implementations. The primary question is whether threads improve multi-query throughput when NumPy releases the GIL; BLAS thread oversubscription may hurt. The sequential loop is the reference; batched BLAS is an additional optimized baseline.

The query vectors are in-corpus vectors for reproducible *computational* comparisons, not a relevance evaluation. Timings exclude embedding generation and file loading. This notebook does not benchmark Phase 1's pure Python TF-IDF implementation.

In [ ]:
%pip -q install threadpoolctl pandas matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, time, platform, statistics
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from threadpoolctl import threadpool_limits, threadpool_info
ROOT=Path('/content/drive/MyDrive/HP-SS-Engine')
EMB=ROOT/'phase2_matrix_ops/2A_sparse_dense/qwen_20k_minilm_384_float32.npy'
OUT=ROOT/'phase3_parallelism/benchmarks'
FIG=ROOT/'phase3_parallelism/figures'
OUT.mkdir(parents=True,exist_ok=True); FIG.mkdir(parents=True,exist_ok=True)
X=np.load(EMB).astype(np.float32,copy=False)
assert X.shape==(20000,384),X.shape
# Normalize once, consistently, before timing. This creates a separate corpus array.
X=np.ascontiguousarray(X/np.maximum(np.linalg.norm(X,axis=1,keepdims=True),1e-12))
print('Corpus',X.shape,'dtype',X.dtype,'CPU count',os.cpu_count(),'BLAS',threadpool_info())

In [ ]:
K=10
# Stable score tie-breaker: smaller document ID first. Same policy in every method.
def topk(scores,k=K):
    idx=np.arange(len(scores))
    order=np.lexsort((idx,-scores))[:k]
    return order.astype(np.int64)
def search_one(q, corpus=X,k=K):
    return topk(corpus @ q,k)
def search_sequential(Q, corpus=X,k=K):
    return [search_one(q,corpus,k) for q in Q]
def search_threaded(Q, corpus=X,k=K,workers=2):
    with ThreadPoolExecutor(max_workers=workers) as pool:
        return list(pool.map(lambda q:search_one(q,corpus,k),Q))
def search_batched(Q, corpus=X,k=K):
    # BLAS matrix multiply: one call for all queries, then stable top-k per row.
    scores=Q @ corpus.T
    return [topk(row,k) for row in scores]

# Verify all three return identical document IDs, not just similar scores.
Q_test=X[[0,31,1024,7777]].copy()
with threadpool_limits(limits=1,user_api='blas'):
    a=search_sequential(Q_test)
    b=search_threaded(Q_test,workers=2)
    c=search_batched(Q_test)
assert all(np.array_equal(u,v) for u,v in zip(a,b))
assert all(np.array_equal(u,v) for u,v in zip(a,c))
print('Correctness PASS for 4 queries; first top-k:',a[0])

In [ ]:
PILOT=True  # Set False for full experiment after reviewing pilot.
N_DOCS=[1000,5000] if PILOT else [1000,5000,10000,20000]
N_QUERIES=[8,32] if PILOT else [16,64,128]
WORKERS=[1,2,4] if PILOT else [1,2,4,8]
BLAS_THREADS=[1] if PILOT else [1,2]
REPEATS=3 if PILOT else 5
SEED=20261020
records=[]
for blas_n in BLAS_THREADS:
  with threadpool_limits(limits=blas_n,user_api='blas'):
    for n in N_DOCS:
      corpus=X[:n]
      for nq in N_QUERIES:
        rng=np.random.default_rng(SEED+n+nq)
        qids=rng.choice(n,size=nq,replace=nq>n)
        Q=np.ascontiguousarray(corpus[qids])
        variants=[('sequential',0,lambda:search_sequential(Q,corpus)),
                  ('batched_blas',0,lambda:search_batched(Q,corpus))]
        variants += [('threadpool',w,lambda w=w:search_threaded(Q,corpus,workers=w)) for w in WORKERS]
        reference=search_sequential(Q,corpus)
        for name,workers,fn in variants:
          result=fn()
          assert all(np.array_equal(u,v) for u,v in zip(reference,result)),(name,n,nq,workers)
          for repeat in range(REPEATS):
            t0=time.perf_counter_ns(); result=fn(); elapsed_ms=(time.perf_counter_ns()-t0)/1e6
            records.append(dict(mode='pilot' if PILOT else 'full',strategy=name,workers=workers,
                blas_threads=blas_n,n_docs=n,n_queries=nq,k=K,repeat=repeat,elapsed_ms=elapsed_ms,
                throughput_qps=nq/(elapsed_ms/1000),cpu_count=os.cpu_count()))
        print('done',blas_n,n,nq)
df=pd.DataFrame(records)
tag='pilot' if PILOT else 'full'
raw_path=OUT/f'phase3a_threadpool_raw_{tag}.csv'
df.to_csv(raw_path,index=False)
print('Saved',len(df),'measured timings:',raw_path)

In [ ]:
group=['mode','strategy','workers','blas_threads','n_docs','n_queries']
sumdf=df.groupby(group,as_index=False).agg(median_ms=('elapsed_ms','median'),mean_ms=('elapsed_ms','mean'),std_ms=('elapsed_ms','std'))
base=sumdf[sumdf.strategy=='sequential'][['blas_threads','n_docs','n_queries','median_ms']].rename(columns={'median_ms':'sequential_median_ms'})
sumdf=sumdf.merge(base,on=['blas_threads','n_docs','n_queries'],how='left')
sumdf['speedup_vs_sequential']=sumdf.sequential_median_ms/sumdf.median_ms
summary_path=OUT/f'phase3a_threadpool_summary_{tag}.csv'
sumdf.to_csv(summary_path,index=False)
print(sumdf.to_string(index=False))
print('Saved:',summary_path)

In [ ]:
fig,ax=plt.subplots(figsize=(10,5))
sub=sumdf[(sumdf.blas_threads==1)&(sumdf.n_docs==max(N_DOCS))&(sumdf.n_queries==max(N_QUERIES))]
labels=[('BLAS batch' if r.strategy=='batched_blas' else f'Thread {int(r.workers)}' if r.strategy=='threadpool' else 'Sequential') for r in sub.itertuples()]
ax.bar(labels,sub.speedup_vs_sequential)
ax.axhline(1,linestyle='--',linewidth=1,color='gray')
ax.set_ylabel('Speedup vs sequential NumPy (median)')
ax.set_title(f'Phase 3A: {max(N_DOCS)} docs × {max(N_QUERIES)} queries, BLAS=1')
ax.tick_params(axis='x',rotation=25)
fig.tight_layout()
figpath=FIG/f'phase3a_threadpool_speedup_{tag}.png'
fig.savefig(figpath,dpi=160);plt.show()
print('Saved figure:',figpath)
print('Interpretation checklist: GIL release; BLAS thread oversubscription; memory bandwidth; Amdahl; worker startup; batch BLAS comparison.')

## What to send back

Run the notebook with `PILOT=True` first. Send the correctness output, summary table and plot. We will inspect them before setting `PILOT=False`. **Do not claim any speedup before execution.** The next report must discuss why threads win/lose, whether speedup is linear, whether batched BLAS is simpler/faster, and why ThreadPool may not be suitable for pure Python loops. Full results and findings will be saved after measured outputs are verified.